# Продажи УТ: выручка, capture и hot reload

Найдём последний месяц с выручкой и предыдущий календарный месяц, сравним обороты по подразделениям, остановим расчёт прироста и изменим его через hot reload.

Метаданные сверены с УТ 11.6.1.61; дата данных ИБ не фиксируется. Используйте отдельную демо-копию и соответствующую ей выгрузку исходников. Подготовка окружения — в [README](../README.md). Сеанс закрывается последней ячейкой; при прерывании выполните `runtime.close()`.

## Подготовка

Укажите `PLATFORM_BIN`, `CONNECTION_STRING` и `SOURCE_ROOT` для отдельной копии УТ 11.6.1.61. В `SOURCE_ROOT` нужна выгрузка исходников этой же конфигурации. Если в ИБ есть пользователи, задайте `username` и `password` в `RuntimeConfig` локально.

In [ ]:
from IPython.display import display
from onec_runtime.config import RuntimeConfig
from onec_runtime.session import ExtensionMode, RuntimeSessionConfig
from onec_runtime_jupyter import InteractiveRuntimeSession

PLATFORM_BIN = r'C:\path\to\1cv8\bin'
CONNECTION_STRING = r'File="C:\demo\UT";'
SOURCE_ROOT = r'C:\exports\UT'
EXTENSION_MODE = ExtensionMode.AUTO  # Совместимое расширение проверяется при запуске.

runtime = InteractiveRuntimeSession.start(
    RuntimeSessionConfig(
        runtime=RuntimeConfig(
            platform_bin=PLATFORM_BIN,
            connection_string=CONNECTION_STRING,
        ),
        source_root=SOURCE_ROOT,
        extension_mode=EXTENSION_MODE,
    )
)

## От какого месяца считать

Берём последнюю дату активного движения с ненулевой выручкой. Сравниваем месяц этой даты с предыдущим календарным месяцем. Если движений нет, ячейка остановится с понятной ошибкой вместо пустой диаграммы.

In [ ]:
%%bsl
ЗапросПоследнейПродажи = Новый Запрос;
ЗапросПоследнейПродажи.Текст =
    "ВЫБРАТЬ ПЕРВЫЕ 1
    |   Движения.Период КАК Период
    |ИЗ
    |   РегистрНакопления.ВыручкаИСебестоимостьПродаж КАК Движения
    |ГДЕ
    |   Движения.Активность
    |   И Движения.СуммаВыручки <> 0
    |УПОРЯДОЧИТЬ ПО
    |   Движения.Период УБЫВ";
ПоследняяПродажа = ЗапросПоследнейПродажи.Выполнить().Выбрать();
Если Не ПоследняяПродажа.Следующий() Тогда
    ВызватьИсключение "В регистре нет активных движений выручки. Выберите ИБ с продажами.";
КонецЕсли;

НачалоПоследнегоМесяца = НачалоМесяца(ПоследняяПродажа.Период);
НачалоПредыдущегоМесяца = ДобавитьМесяц(НачалоПоследнегоМесяца, -1);
КонецПоследнегоМесяца = КонецМесяца(ПоследняяПродажа.Период);

## Обороты регистра

Виртуальная таблица `Обороты` учитывает движения выручки, в том числе корректировки и возвраты. Группируем по месяцу и подразделению; `СуммаВыручки` — ресурс регистра с НДС. В этой выборке нет пересчёта сумм документов по валютам.

In [ ]:
%%bsl
ЗапросПродаж = Новый Запрос;
ЗапросПродаж.Текст =
    "ВЫБРАТЬ
    |   Продажи.Период КАК Месяц,
    |   Продажи.Подразделение КАК Подразделение,
    |   СУММА(Продажи.СуммаВыручкиОборот) КАК Выручка
    |ИЗ
    |   РегистрНакопления.ВыручкаИСебестоимостьПродаж.Обороты(
    |       &НачалоПериода, &КонецПериода, Месяц, ) КАК Продажи
    |СГРУППИРОВАТЬ ПО
    |   Продажи.Период,
    |   Продажи.Подразделение
    |УПОРЯДОЧИТЬ ПО
    |   Месяц";
ЗапросПродаж.УстановитьПараметр("НачалоПериода", НачалоПредыдущегоМесяца);
ЗапросПродаж.УстановитьПараметр("КонецПериода", КонецПоследнегоМесяца);
ПродажиПоМесяцам = ЗапросПродаж.Выполнить().Выгрузить();
Если ПродажиПоМесяцам.Количество() = 0 Тогда
    ВызватьИсключение "За выбранные два месяца оборотов выручки нет.";
КонецЕсли;

## Таблица и диаграммы

Переносим результат `%%bsl` в pandas. Подписи ссылок получаем через `refs="presentation"`; они нужны только для показа, а не для сопоставления объектов. Месяц без движений показываем с нулевой выручкой.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

sales = ПродажиПоМесяцам.to_df(refs='presentation')
display(sales)
monthly = (sales.assign(Месяц=pd.to_datetime(sales['Месяц']).dt.to_period('M'))
           .groupby('Месяц')['Выручка'].sum())
months = pd.period_range(end=monthly.index.max(), periods=2, freq='M')
monthly = monthly.reindex(months, fill_value=0)
display(monthly.rename_axis('Месяц').to_frame())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(month) for month in monthly.index], monthly.map(float))
ax.set_ylabel('Выручка с НДС')
ax.set_title('Два последних календарных месяца по данным ИБ')
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
latest_month = sales['Месяц'].max()
by_department = (sales.loc[sales['Месяц'] == latest_month]
                 .assign(Подразделение=lambda data:
                         data['Подразделение'].fillna('Без подразделения'))
                 .groupby('Подразделение')['Выручка'].sum()
                 .sort_values(ascending=False).head(10))
display(by_department.to_frame('Выручка'))

fig, ax = plt.subplots(figsize=(9, max(3, len(by_department) * 0.4)))
ax.barh(by_department.index, by_department.map(float))
ax.invert_yaxis()
ax.set_xlabel('Выручка с НДС')
ax.set_title(f'Подразделения за {str(latest_month)[:7]}')
fig.tight_layout()
display(fig)
plt.close(fig)

## Типовая логика прироста

Сначала суммируем выручку в BSL и вызываем `ПродажиСервер.ПроцентПрироста`. В УТ 11.6.1.61 эта функция отдельно обрабатывает нулевую и отрицательную базу сравнения: результат для нулевой базы не равен обычному математическому проценту роста.

In [ ]:
%%bsl
ВыручкаПредыдущегоМесяца = 0;
ВыручкаПоследнегоМесяца = 0;
Для Каждого СтрокаПродаж Из ПродажиПоМесяцам Цикл
    Если СтрокаПродаж.Месяц = НачалоПоследнегоМесяца Тогда
        ВыручкаПоследнегоМесяца = ВыручкаПоследнегоМесяца + СтрокаПродаж.Выручка;
    Иначе
        ВыручкаПредыдущегоМесяца = ВыручкаПредыдущегоМесяца + СтрокаПродаж.Выручка;
    КонецЕсли;
КонецЦикла;

ПриростВыручки = ПродажиСервер.ПроцентПрироста(
    ВыручкаПредыдущегоМесяца, ВыручкаПоследнегоМесяца);
КонтрольныйПриростДо = ПродажиСервер.ПроцентПрироста(0, 100);

In [ ]:
growth_before = ПриростВыручки.materialize()
probe_before = КонтрольныйПриростДо.materialize()
print('Прирост выручки, %:', growth_before)
print('Контрольный вызов (0, 100), %:', probe_before)

## Останов перед возвратом

Поставим точку на строке 9311 метода `ПродажиСервер.ПроцентПрироста`, перед `Возврат Результат;`. Номер относится к выгрузке УТ 11.6.1.61.

In [ ]:
runtime.add_capture_point(r'CommonModules\ПродажиСервер\Ext\Module.bsl', 9311)

In [ ]:
%%bsl
Результат = ПродажиСервер.ПроцентПрироста(
    ВыручкаПредыдущегоМесяца, ВыручкаПоследнегоМесяца);

In [ ]:
status = runtime.status()
assert status.state.value == 'captured'
stack = runtime.runtime_api.capture_stack(cursor=0, limit=20)
display(pd.DataFrame(stack['frames'])[['level', 'module_type', 'line']])

## Значения внутри вызова

Пока вызов остановлен, `КонтекстОтладки` даёт аргументы и локальный результат. Скопируем только три числа в таблицу для просмотра в Python.

In [ ]:
%%bsl
СнимокПрироста = Новый ТаблицаЗначений;
СнимокПрироста.Колонки.Добавить("ВыручкаПредыдущегоМесяца");
СнимокПрироста.Колонки.Добавить("ВыручкаПоследнегоМесяца");
СнимокПрироста.Колонки.Добавить("ПриростПроцентов");
СтрокаСнимка = СнимокПрироста.Добавить();
СтрокаСнимка.ВыручкаПредыдущегоМесяца = КонтекстОтладки.ПредыдущееЗначение;
СтрокаСнимка.ВыручкаПоследнегоМесяца = КонтекстОтладки.ТекущееЗначение;
СтрокаСнимка.ПриростПроцентов = КонтекстОтладки.Результат;

In [ ]:
display(СнимокПрироста.to_df())

In [ ]:
completed = runtime.resume_capture()
runtime.clear_capture_points()
assert completed.succeeded and completed.state.value == 'completed'
print('Результат после продолжения, %:', completed.result)

## Hot reload: нет базы для процента

Типовая функция возвращает 100% для пары `(0, 100)`. Попробуем другую трактовку: при нулевой предыдущей и положительной текущей выручке процент не определён. В **локальной копии** выгрузки `SOURCE_ROOT` откройте `CommonModules/ПродажиСервер/Ext/Module.bsl`. В функции `ПроцентПрироста` найдите ветку `ИначеЕсли ПредыдущееЗначение = 0 Тогда` и замените в ней `Результат = 100;` на `Результат = Неопределено;`. После правки ветка должна выглядеть так:

```bsl
ИначеЕсли ПредыдущееЗначение = 0 Тогда
    Результат = Неопределено;
```

Сохраните файл, оставив остальную функцию без изменений. Конфигурация ИБ не меняется: следующий шаг загрузит изменённый модуль в текущий runtime-сеанс. При переносе в Python значение `Неопределено` представлено как `ONEC_UNDEFINED`.

In [ ]:
runtime.load_worker_module(r'CommonModules\ПродажиСервер\Ext\Module.bsl')

In [ ]:
%%bsl
ПриростПослеПерезагрузки = ПродажиСервер.ПроцентПрироста(
    ВыручкаПредыдущегоМесяца, ВыручкаПоследнегоМесяца);
КонтрольныйПриростПосле = ПродажиСервер.ПроцентПрироста(0, 100);

In [ ]:
from onec_runtime.value_materialization import ONEC_UNDEFINED

growth_after = ПриростПослеПерезагрузки.materialize()
probe_after = КонтрольныйПриростПосле.materialize()
assert probe_before == 100 and probe_after is ONEC_UNDEFINED
print('Контрольный вызов (0, 100):', probe_before, '→ Неопределено')
print('Выручка ИБ:', growth_before, '→',
      'Неопределено' if growth_after is ONEC_UNDEFINED else growth_after)

## Что дальше

Можно заменить два месяца другим интервалом, добавить отбор по организации или исследовать отдельное подразделение. Для сравнения с ЗУП откройте [01-overview.ipynb](../ZUP/01-overview.ipynb).

## Завершение

In [ ]:
runtime.close()
print('Сеанс закрыт')